![Week 6 Day 3](../assets/banners/w6d3.png)

# Week 6 Day 3 - Streaming Progress, the Report Viewer and Chat With Results

Yesterday Asha got her dashboard, and you used the review queue to override a mark and resume a batch. But on Day 1 we followed batches the clumsy way: asking "are you done yet?" five times a second.

Today the progress comes to us. We'll take the events that LangGraph nodes already emit, carry them across HTTP with **Server-Sent Events (SSE)**, and watch a batch grade itself live, right here in the notebook. Then we close the loop on the teacher's side of the product: the **report-card viewer** and **chat with the results**.

![Week 6 roadmap](../assets/roadmap/week6.png)

### 🎯 By the end of this lab you will
1. ⚖️ Choose between polling, SSE and WebSockets, and know why GradeWise uses SSE
2. 📡 Emit custom progress events from a LangGraph node and stream them with `stream_mode="custom"`
3. 🧵 Write an SSE endpoint from scratch, and read the wire format with your own eyes
4. 🚀 Watch a real GradeWise batch stream live, reconnect without missing an event, and draw live progress bars
5. 📊 View report cards (HTML and PDF) and ask the results agent questions through the API

### 🎬 Videos for this lab

| # | Lecture | Type |
|:-:|---|:-:|
| 1 | Day 3 - Streaming Grading Progress from LangGraph to the UI | 🧪 |
| 2 | Day 3 - Report Card Viewer and Chat With Results | 🧪 |

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">Everything today is <b>offline and free</b> (🧪 demo mode). Today's sandbox is <code>output/week6/day3/</code>. If a report card shows up as raw HTML, trust the notebook (JupyterLab: <i>File → Trust Notebook</i>; VS Code trusts it when you run it).</span>
        </td>
    </tr>
</table>

In [ ]:
# Imports and environment first, all in one place

import os

from dotenv import load_dotenv
from IPython.display import HTML, Markdown, display

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")

if openai_api_key:
    print(f"✅ OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("ℹ️ No OpenAI API Key set. That's fine this week: every cell runs offline in 🧪 demo mode.")
    print("   (To grade for real later, see Part 5 of the setup guide, or setup/SWAP_PROVIDER.md)")

## Part 1: Polling, SSE or WebSockets?

There are three classic ways for a browser to learn that something changed on the server:

| | How it works | Good at | Costs |
|---|---|---|---|
| **Polling** | Client asks `GET /status` every second | Dead simple; works everywhere | Wasted requests; updates arrive up to one interval late |
| **Server-Sent Events** | One long `GET`; the server writes events down it as they happen | One-way server → client updates; auto-reconnect built into browsers; plain HTTP | Server → client only |
| **WebSockets** | A two-way socket upgraded from HTTP | Chat, games, collaborative editing | More moving parts; proxies and hosts need care |

Grading progress only flows one way (server → Asha), so **SSE** is the sweet spot. And because it's plain HTTP, it goes through the same proxies, load balancers and auth as every other request.

## Part 2: Step one: progress events out of LangGraph

You met this in week 5, briefly: any node can write to LangGraph's **custom stream** with `get_stream_writer()`, and whoever streams the graph with `stream_mode="custom"` receives those writes *while the graph runs*. Here's the idea in a tiny graph:

In [ ]:
import time
from typing import TypedDict

from langgraph.config import get_stream_writer
from langgraph.graph import END, START, StateGraph


class MiniState(TypedDict):
    marks: list[float]


def grade(state: MiniState) -> dict:
    writer = get_stream_writer()  # 💡 only works while the graph is running this node
    marks = []
    for i, mark in enumerate([2, 3, 1.5], start=1):
        time.sleep(0.3)  # pretend this is a model call
        marks.append(mark)
        writer({"stage": "grade_question", "message": f"question {i} of 3 graded"})
    return {"marks": marks}


def report(state: MiniState) -> dict:
    get_stream_writer()({"stage": "write_report", "message": f"total {sum(state['marks']):g}"})
    return {}


builder = StateGraph(MiniState)
builder.add_node("grade", grade)
builder.add_node("report", report)
builder.add_edge(START, "grade")
builder.add_edge("grade", "report")
builder.add_edge("report", END)
mini_graph = builder.compile()

start = time.perf_counter()
for event in mini_graph.stream({"marks": []}, stream_mode="custom"):
    print(f"{time.perf_counter() - start:4.1f}s  {event}")

Each event arrived **as it happened**, not at the end. That's all GradeWise does: every orchestrator node calls `emit(stage, student_id, message, ...)` (`gradewise/grading/events.py`), a three-line wrapper around exactly this writer. `grade_batch(..., on_event=callback)` then streams the graph and calls your callback for every event.

## Part 3: Step two: Server-Sent Events, by hand

SSE is a wonderfully simple text format. The response has the content type `text/event-stream`, and each event is a few `field: value` lines followed by a **blank line**:

```text
id: 7
event: progress
data: {"stage": "grade_question", "message": "question 1 of 3 graded"}

```

Let's put our mini graph behind an SSE endpoint. A **generator** yields the text, and FastAPI's `StreamingResponse` sends each piece the moment it's yielded:

In [ ]:
import json

from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from fastapi.testclient import TestClient

sse_app = FastAPI()


@sse_app.get("/events")
def events() -> StreamingResponse:
    def stream():
        for n, event in enumerate(mini_graph.stream({"marks": []}, stream_mode="custom"), start=1):
            yield f"id: {n}\nevent: progress\ndata: {json.dumps(event)}\n\n"
        yield 'event: end\ndata: {"status": "done"}\n\n'

    return StreamingResponse(stream(), media_type="text/event-stream")


start = time.perf_counter()
with TestClient(sse_app).stream("GET", "/events") as response:
    print("content-type:", response.headers["content-type"], "\n")
    for line in response.iter_lines():
        print(f"{time.perf_counter() - start:4.1f}s | {line}")

That's the raw wire format, arriving line by line. A client needs about ten lines to parse it: remember the `event:` name, collect `data:` lines, and when a blank line arrives, the event is complete. Open `gradewise/ui/client.py` and find `stream_events()`: that's exactly what the dashboard does.

## Part 4: GradeWise's stream, end to end

Here's the path a progress event takes in GradeWise:

![Diagram](../assets/diagrams/b30ec5ba6498.png)

<details><summary>🔎 Diagram source (Mermaid)</summary>

```text
sequenceDiagram
  participant N as Orchestrator node
  participant J as BatchJob (jobs.py)
  participant E as GET /batches/{id}/events
  participant U as Dashboard (client.py)
  N->>J: emit() → on_event(stage, student, detail)
  J->>J: number it (id 1, 2, 3 …), update per-student progress
  U->>E: GET … (Last-Event-ID: 41)
  loop every 0.25 s
    E->>J: events_after(41)
    E-->>U: id / event: progress / data: {...}
  end
  E-->>U: event: end {"status": "paused"}
```

</details>


Two design choices make it robust:
- **Events are stored, then streamed.** The worker thread appends numbered events to the job; the SSE endpoint *replays* from any point. A browser that connects late, or reconnects, misses nothing.
- **The stream ends when the batch stops** (paused, done or failed) with one final `event: end`. After a review, the dashboard simply connects again.

Let's see it. A demo API for this notebook, the course notes ingested (so the grader has evidence to cite), and then a batch of three students:

In [ ]:
import shutil
import time

from fastapi.testclient import TestClient

from gradewise.api.main import create_app
from gradewise.config import OUTPUT_DIR
from gradewise.console import success

SANDBOX = OUTPUT_DIR / "week6" / "day3"
shutil.rmtree(SANDBOX, ignore_errors=True)  # 💡 a fresh start every time you run the notebook

app = create_app(SANDBOX, demo=True)  # 🧪 demo=True: offline fakes, no API key, no cost
client = TestClient(app)  # 💡 calls the app in-process: real HTTP semantics, no server, no port


def show(data, limit: int = 1500) -> None:
    """Pretty-print JSON, cut short so a big response doesn't flood the notebook."""
    text = json.dumps(data, indent=2, ensure_ascii=False, default=str)
    print(text if len(text) <= limit else text[:limit] + "\n  …")


def wait_for(batch_id: str, *states: str, timeout: float = 120) -> dict:
    """Poll GET /batches/{id} until the batch reaches one of `states` (default: paused, done or failed)."""
    states = states or ("paused", "done", "failed")
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        batch = client.get(f"/batches/{batch_id}").json()
        if batch["status"] in states:
            return batch
        time.sleep(0.2)
    raise TimeoutError(f"{batch_id} never reached {states}")


success("A private GradeWise API for this notebook", {"Mode": "🧪 demo (offline fakes)", "Writes to": str(SANDBOX)})

In [ ]:
kb = client.post("/knowledge-bases/cs201/documents", data={"include_course_notes": "true"}).json()
print(f"📚 {kb['name']}: {kb['chunks']} chunks")

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/stop.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ef4444;">Stop & think</h2>
            <span style="color:#ef4444;">We're about to start a 3-student batch and stream its events. The exam has 15 questions. Before running, predict: roughly how many <code>grade_question</code> events will arrive, and what will the very last event be: <code>done</code>, or something else?</span>
        </td>
    </tr>
</table>

In [ ]:
from gradewise.ui.common import ICONS  # 💡 the same stage icons the dashboard's event log uses

batch_id = client.post("/batches", data={"exam_id": "cs_midterm", "from_transcripts": "true", "students": "3"}).json()[
    "batch_id"
]


def read_sse(lines):
    """Turn SSE lines into (event_name, data) pairs, exactly like gradewise/ui/client.py."""
    name, data = "message", []
    for line in lines:
        if line.startswith(":"):
            continue  # 💡 a comment line: the server's keep-alive
        if line.startswith("event:"):
            name = line[6:].strip()
        elif line.startswith("data:"):
            data.append(line[5:].strip())
        elif not line and data:
            yield name, json.loads("\n".join(data))
            name, data = "message", []


start, graded, last_id = time.perf_counter(), 0, 0
with client.stream("GET", f"/batches/{batch_id}/events") as response:
    for name, event in read_sse(response.iter_lines()):
        if name == "end":
            print(f"\n🔚 end of stream: {event}")
            break
        last_id = event["id"]
        if event["stage"] == "grade_question":  # 💡 45 of these: show them as dots
            graded += 1
            print("·", end="", flush=True)
            continue
        who = event["student_id"] or "batch"
        pct = f"{event['progress']['percent']:>3}%" if event.get("progress") else "    "
        print(
            f"\n{time.perf_counter() - start:5.2f}s #{event['id']:<3} {ICONS.get(event['stage'], '▸')} "
            f"{event['stage']:<13} {who:<11} {pct} {event['message'][:60]}",
            end="",
        )
print(f"\n{graded} grade_question events · last event id {last_id}")

It ended with `paused`, not `done`: the stream stops whenever the batch stops, and the batch stopped because the first flagged student is waiting for Asha.

### Reconnecting without missing anything
Networks drop. When a browser's `EventSource` reconnects, it sends the last ID it saw in a `Last-Event-ID` header, and our endpoint replays only what came after. (`?after=N` does the same for clients that can't set headers.) Watch:

In [ ]:
def count_events(**kwargs) -> int:
    with client.stream("GET", f"/batches/{batch_id}/events", **kwargs) as response:
        return sum(1 for name, _ in read_sse(response.iter_lines()) if name == "progress")


print("from the start           :", count_events(), "events")
print("Last-Event-ID: 50        :", count_events(headers={"Last-Event-ID": "50"}), "events")
print(f"after={last_id} (all seen)    :", count_events(params={"after": last_id}), "events")

## Part 5: Live progress bars, the way the dashboard draws them

The dashboard's `live_progress()` (`gradewise/ui/common.py`) keeps one placeholder per student and **overwrites it in place** as events arrive. A notebook can do the same with a display handle: `display(..., display_id=True)` returns a handle whose `.update()` redraws that output.

Let's approve each paused student and follow the batch to the end, with live bars:

In [ ]:
from html import escape


def bars_html(progress: dict, log: list[str]) -> str:
    rows = "".join(
        f"<div style='margin:4px 0'><code>{escape(sid)}</code> {escape(p['stage'])} "
        f"<div style='background:#262F55;border-radius:6px;height:10px;width:320px'>"
        f"<div style='background:linear-gradient(90deg,#7C5CFF,#22D3EE);height:10px;border-radius:6px;"
        f"width:{min(100, p['percent']) * 3.2:.0f}px'></div></div></div>"
        for sid, p in progress.items()
    )
    tail = "<br>".join(escape(line) for line in log[-6:])
    return f"<div style='font-family:monospace;font-size:12px'>{rows}<div style='color:#9AA3C7'>{tail}</div></div>"


batch = client.get(f"/batches/{batch_id}").json()
progress = {p["student_id"]: p for p in batch["progress"]}
log: list[str] = []
handle = display(HTML(bars_html(progress, log)), display_id=True)

while batch["status"] == "paused":
    client.post(f"/batches/{batch_id}/review", json={"approved": True}).raise_for_status()  # 🧑‍🏫 approve as-is
    with client.stream("GET", f"/batches/{batch_id}/events", params={"after": last_id}) as response:
        for name, event in read_sse(response.iter_lines()):
            if name == "end":
                break
            last_id = event["id"]
            if event.get("progress"):
                progress[event["student_id"]] = event["progress"]
            if event["stage"] == "done":
                for p in progress.values():
                    p.update(stage="done", percent=100)
            icon = ICONS.get(event["stage"], "▸")
            log.append(f"{icon} {event['stage']} {event['student_id'] or ''} {event['message'][:50]}")
            handle.update(HTML(bars_html(progress, log)))
    batch = wait_for(batch_id)

print(f"Batch {batch['status']} after {last_id} events")

<img src="../assets/week6/04_grade_batch.png" width="820" alt="✍️ The same stream in the dashboard: a bar per student and an event log, updated in place." style="border-radius:10px;border:1px solid #262F55" />

*✍️ The same stream in the dashboard: a bar per student and an event log, updated in place.*

## Part 6: The report-card viewer

Grading is done; now the teacher's reward. The 📊 Reports page shows the class first, then any student's card. Both come from two endpoints you met on Day 1.

### The class overview
`GET /reports/{exam_id}` computes every number in code (means, medians, bands, hardest questions): **numbers come from code, never from a model's arithmetic.** Here it is as a tiny HTML bar chart, the same idea as `styles.bar_chart()`:

In [ ]:
overview = client.get("/reports/cs_midterm").json()
print(
    f"{overview['count']} students · mean {overview['mean_percent']}% · median {overview['median_percent']}% · "
    f"bands {overview['bands']} · still flagged {overview['flagged']}"
)

bars = "".join(
    f"<div style='display:flex;align-items:center;gap:8px;margin:3px 0;font-family:sans-serif;font-size:12px'>"
    f"<span style='width:44px'>{q['question_id']}</span>"
    f"<span style='background:linear-gradient(90deg,#7C5CFF,#22D3EE);height:12px;border-radius:4px;"
    f"width:{q['mean_percent'] * 2.4:.0f}px'></span><span>{q['mean_percent']:g}%</span></div>"
    for q in overview["questions"][:6]
)
display(HTML(f"<b>🧗 Hardest questions (class mean %)</b>{bars}"))

<img src="../assets/week6/06_reports.png" width="820" alt="📊 Reports in the dashboard: KPIs, the mark distribution, the hardest questions and the student table." style="border-radius:10px;border:1px solid #262F55" />

*📊 Reports in the dashboard: KPIs, the mark distribution, the hardest questions and the student table.*

### One student's report card
`GET /reports/{exam_id}/{student_id}?format=html` returns the same self-contained HTML card that week 5 wrote, and `?format=pdf` a PDF of it. Let's show the HTML inline and save the PDF:

In [ ]:
import html as html_lib

student = overview["students"][0]["student_id"]
card_html = client.get(f"/reports/cs_midterm/{student}", params={"format": "html"}).text

# 💡 Inside an <iframe srcdoc>, so the card's CSS can't restyle this notebook.
frame = f'<iframe srcdoc="{html_lib.escape(card_html)}" style="width:100%;height:760px;border:0"></iframe>'
display(HTML(f"<div>{frame}</div>"))

In [ ]:
pdf = client.get(f"/reports/cs_midterm/{student}", params={"format": "pdf"})
pdf_path = SANDBOX / f"{student}.pdf"
pdf_path.write_bytes(pdf.content)
print(f"📄 {pdf.headers['content-type']} · {len(pdf.content) / 1024:.0f} KB · starts {pdf.content[:5]}")
print(f"   saved to {pdf_path}")

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/important.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Important</h2>
            <span style="color:#ff7800;">A report card contains <b>model-written text</b> (feedback, justifications) and student-written text (quotes from answers). Treat both as untrusted. The card escapes everything it renders, and the dashboard shows it in an <code>&lt;iframe&gt;</code> with a <code>data:</code> URL, which gives it an opaque origin: even if something slipped through, nothing inside the card could script the dashboard around it. A student who writes <code>&lt;script&gt;</code> in an exam answer is a real threat model!</span>
        </td>
    </tr>
</table>

## Part 7: Chat with the results

Week 5's results agent answers the questions a teacher asks after a marking session, with four tools: exact class statistics, semantic search over graded answers, one student's report, and the list of flags. The API exposes it as `POST /ask`; the 💬 Chat page is a `st.chat_input` and `st.chat_message` loop around it.

In demo mode, `DemoResultsModel` picks the tool by keyword and shows its raw output (honestly labelled). With a real model, the same tool output becomes a sentence or two of grounded prose.

In [ ]:
for question in [
    "Which question did the class struggle with most?",
    f"Why did {student} lose marks?",
    "Which answers still need my review?",
]:
    answer = client.post("/ask", json={"question": question, "exam_id": "cs_midterm"}).json()["answer"]
    display(Markdown(f"**🧑‍🏫 {question}**\n\n{answer[:900]}"))

<img src="../assets/week6/07_chat.png" width="820" alt="💬 Chat with results: example questions, then the agent's answer built from its tools." style="border-radius:10px;border:1px solid #262F55" />

*💬 Chat with results: example questions, then the agent's answer built from its tools.*

And when there's nothing to talk about yet, the API says so politely (a `404` with the friendly fields, not a stack trace):

In [ ]:
r = client.post("/ask", json={"question": "How did they do?", "exam_id": "no_such_exam"})
print(r.status_code, r.json()["error"], "·", r.json()["fix"])

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/business.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#22c55e;">Business value</h2>
            <span style="color:#22c55e;">Live progress isn't decoration. A teacher who sees "student_07 · grading 9/15" trusts the system and gets on with something else; one who stares at a spinner for three minutes refreshes the page, double-submits, and files a support ticket. And chat over results turns a marking session into <b>insight</b>: "who struggled with subnetting?" is exactly the question that shapes next week's lesson, and it's where EdTech products earn renewals.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/gradewise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#7c5cff;">GradeWise</h2>
            <span style="color:#7c5cff;">The stream is spread over four small pieces: <code>emit()</code> in <code>gradewise/grading/events.py</code> (nodes report progress), <code>grade_batch(on_event=...)</code> in <code>gradewise/orchestrator.py</code>, <code>BatchJob.on_event()</code> / <code>events_after()</code> in <code>gradewise/api/jobs.py</code>, and the <code>batch_events()</code> route in <code>gradewise/api/main.py</code>. On the UI side: <code>client.stream_events()</code> and <code>common.live_progress()</code>. The report viewer and chat are <code>gradewise/ui/pages/reports.py</code> and <code>chat.py</code>, on top of week 5's <code>gradewise/reports/</code>.</span>
        </td>
    </tr>
</table>

## Recap, and where we are heading

Today you:
- ⚖️ compared polling, SSE and WebSockets, and picked SSE for one-way progress
- 📡 streamed custom events out of a LangGraph node with `get_stream_writer()` and `stream_mode="custom"`
- 🧵 wrote an SSE endpoint by hand and read the wire format line by line
- 🚀 watched a GradeWise batch stream live, reconnected with `Last-Event-ID`, and drew live progress bars
- 📊 viewed a report card inline and as a PDF, and chatted with the results

**Tomorrow (Day 4):** one command to run it all. We package the API and the dashboard into a **Docker** image, wire them together with **Docker Compose**, and look at three ways to put GradeWise on the internet.

⏭ **Next:** `week6/day4.ipynb`

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/exercise.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f59e0b;">Exercise</h2>
            <span style="color:#f59e0b;">Build a <b>resilient progress watcher</b>. Write <code>watch(batch_id)</code> that follows a batch's SSE stream and prints one text progress bar per student (e.g. <code>student_02 ████████░░ 80% grade</code>), redrawn in place with a display handle. Then make it survive a dropped connection: stop reading after 10 events (simulating a network blip), reconnect with the <code>Last-Event-ID</code> header, and prove that no event was lost and none was seen twice (compare the IDs). Solution: <code>week6/solutions/day3_SOLUTION.ipynb</code>.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 110px; height: 110px; vertical-align: middle;">
            <img src="../assets/callouts/resources.png" width="100" height="100" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#3b82f6;">Resources</h2>
            <span style="color:#3b82f6;">• MDN: Using server-sent events: <a href="https://developer.mozilla.org/en-US/docs/Web/API/Server-sent_events/Using_server-sent_events">developer.mozilla.org: Server-sent events</a><br/>• LangGraph streaming (custom data, subgraphs): <a href="https://docs.langchain.com/oss/python/langgraph/streaming">docs.langchain.com/oss/python/langgraph/streaming</a><br/>• FastAPI <code>StreamingResponse</code>: <a href="https://fastapi.tiangolo.com/advanced/custom-response/">fastapi.tiangolo.com/advanced/custom-response</a></span>
        </td>
    </tr>
</table>